# NanoStream-OD vs YOLOv8n vs FOMO on a T4 GPU

Trains the full NanoStream-OD tier family (**mcu / pro / gpu**) plus the **YOLOv8n** and **FOMO-style** baselines on the synthetic 4-class shapes+faces benchmark (circle, square, triangle, **face**), then evaluates everything on the same held-out validation split and renders face-detection demo images.

Tiers:
- **mcu** - 160 px, shift-only, <256 KB SRAM MCU target (the C-exportable model)
- **pro** - 256 px, laptop target
- **gpu** - 320 px, server/industry target

Run the cells top to bottom. Every training cell uses the T4 GPU.

In [ ]:
# 1. GPU check + clone the repo (edit the URL to your fork if needed)
!nvidia-smi

!git clone https://github.com/Flaxmbot/nanostream.git || true
%cd nanostream
!git pull || true

In [ ]:
# 2. Install dependencies (torch + CUDA are preinstalled on Colab)
!pip install -q opencv-python ultralytics pytest
!python -c "import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())"

## Train NanoStream-OD tiers

All tiers use the YOLO-grade augmentation pipeline (mosaic, mixup, multi-scale, geometric, photometric, cutout) wired into `benchmarks/train_nanostream.py`.

In [ ]:
# 3a. NanoStream-OD mcu: 160 px, shift-only, the MCU artifact
!python -m benchmarks.train_nanostream --profile mcu --steps 3000 --batch 64 --data_len 1200 --device cuda --out benchmarks/runs/ckpt

In [ ]:
# 3b. NanoStream-OD pro: 256 px laptop tier
!python -m benchmarks.train_nanostream --profile pro --steps 2000 --batch 32 --data_len 1200 --device cuda --out benchmarks/runs/ckpt

In [ ]:
# 3c. NanoStream-OD gpu: 320 px server/industry tier
!python -m benchmarks.train_nanostream --profile gpu --steps 1500 --batch 16 --data_len 1200 --device cuda --out benchmarks/runs/ckpt

## Train the baselines

In [ ]:
# 4a. FOMO-style baseline (same augmentation recipe, 160 px)
!python -m benchmarks.train_fomo --steps 1000 --batch 64 --data_len 1200 --device cuda --out benchmarks/runs/ckpt

In [ ]:
# 4b. YOLOv8n baseline: export the dataset to YOLO format, then train (pretrained, 160 px)
#     Add --from_scratch to train YOLO from random init for an apples-to-apples comparison.
!python -m benchmarks.run_compare data --train_len 1200
!python -m benchmarks.run_compare train-yolo --yolo_model n --imgsz 160 --yolo_epochs 40 --yolo_batch 64

## Evaluate everything

In [ ]:
# 5. Unified AP / latency / size comparison on the same 60-image val split
!python -m benchmarks.run_compare eval

In [ ]:
# 6. Show the results table + face-demo images
import json, pathlib
from IPython.display import Image, display

res = json.loads(pathlib.Path("benchmarks/runs/results.json").read_text())
print(f"{'model':<18} {'mAP50':>7} {'mAP50:95':>9} {'P':>6} {'R':>6} {'params':>9} {'RAM(KB)':>8}")
for name, r in res.items():
    ram = r.get("static_ram_bytes", 0) / 1024
    print(f"{name:<18} {r['mAP50']:7.3f} {r['mAP50_95']:9.3f} "
          f"{r['precision']:6.3f} {r['recall']:6.3f} {r['params']:9d} {ram:8.1f}")

print("\nFace demo images (green = NanoStream boxes):")
for p in sorted(pathlib.Path("benchmarks/runs/face_demo").glob("*.png")):
    display(Image(filename=str(p)))

## MCU export check (mcu tier only)

Exports the trained mcu model to a static C header and verifies the <256 KB SRAM budget by compiling and running the C kernel.

In [ ]:
# 7. Export mcu -> C header, compile the C kernel, check static BSS < 256 KB
!python -m nanostream.export_cli --model benchmarks/runs/ckpt/nanostream_mcu.pt --out nanostream/mcu/model_weights.h --calib-samples 8
!python -m pytest tests/test_export.py -q